# ATAC-seq trajectory archetypes and histone mark subclusters

In [ ]:
from pathlib import Path

import datashader as ds
import datashader.transfer_functions as tf
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

## Config + load

In [ ]:
# Resolve this analysis folder (paper/02_atac) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '01_ccre_insertion_matrix.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/02_atac')

DIR = _here()
CLUST = DIR / 'results' / 'ccre_histsub_clusters.parquet'   # <- 08_histsub_clusters.py
if not CLUST.exists():
    raise SystemExit(f'missing {CLUST} — run 08_histsub_clusters.py first')
FIG_DIR = DIR / 'figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ['ESC', 'DE', 'HB', 'iHEP', 'mHEP']
HIST = ['H3K4me3', 'H3K27ac', 'H3K27me3', 'H3K9me3']
ZLIM = 2.5
TAB10 = plt.get_cmap('tab10').colors

df = pl.read_parquet(CLUST)
arch = df['atac_cluster'].to_numpy()
sub = df['hist_subcluster'].to_numpy()
n_arch = int(arch.max())
atac = np.column_stack([df[f'atac_{s}'].to_numpy() for s in STAGES])
hist = np.column_stack([df[f'log2fc_{m}.{s}'].to_numpy() for m in HIST for s in STAGES])
histz = (hist - hist.mean(0)) / hist.std(0)        # column z (magnitude-preserving)
sizes = [int((arch == c).sum()) for c in range(1, n_arch + 1)]
starts = np.concatenate([[0], np.cumsum(sizes)])
print(f'{df.height:,} cCREs; {n_arch} archetypes; sizes {sizes}')

## Figure 1 — tiled heatmap (separate per-archetype axes)

In [ ]:
a_lo, a_hi = np.percentile(atac, [2, 98])
panel_cols = ['ATAC'] + HIST
GAP = 0.05


def strip(ax, seg):
    ax.set_xlim(0, 1); ax.set_ylim(len(seg) - 0.5, -0.5)
    i = 0
    while i < len(seg):
        j = i
        while j < len(seg) and seg[j] == seg[i]:
            j += 1
        c = (0.7, 0.7, 0.7) if seg[i] >= 99 else TAB10[int(seg[i]) % 10]
        ax.add_patch(Rectangle((0, i - 0.5), 1, j - i, facecolor=c, edgecolor='none'))
        i = j
    ax.axis('off')


fig = plt.figure(figsize=(2 + 1.9 * len(panel_cols), 11))
# rows: n_arch bands + spacer + colorbar (spacer keeps colorbars off the x labels)
gs = fig.add_gridspec(n_arch + 2, 2 + len(panel_cols),
                      height_ratios=sizes + [max(sizes) * 0.12, max(sizes) * 0.04],
                      width_ratios=[0.55, 0.3] + [5] * len(panel_cols),
                      hspace=GAP, wspace=0.06)

for ci in range(n_arch):
    a, b = starts[ci], starts[ci + 1]
    axl = fig.add_subplot(gs[ci, 0]); axl.axis('off')
    axl.text(0.5, 0.5, f'cl{ci+1}\nn={sizes[ci]:,}', ha='center', va='center',
             fontsize=8, fontweight='bold')
    strip(fig.add_subplot(gs[ci, 1]), sub[a:b])
    for pj, name in enumerate(panel_cols):
        ax = fig.add_subplot(gs[ci, pj + 2])
        if name == 'ATAC':
            ax.imshow(atac[a:b], aspect='auto', cmap='cividis', vmin=a_lo, vmax=a_hi,
                      interpolation='nearest', rasterized=True)
        else:
            mi = HIST.index(name)
            ax.imshow(histz[a:b, mi * 5:(mi + 1) * 5], aspect='auto', cmap='RdBu_r',
                      vmin=-ZLIM, vmax=ZLIM, interpolation='nearest', rasterized=True)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():      # remove black panel borders
            sp.set_visible(False)
        if ci == 0:
            ax.set_title(name, fontsize=9)
        if ci == n_arch - 1:
            ax.set_xticks(range(5)); ax.set_xticklabels(STAGES, fontsize=6.5, rotation=90)

cbrow = n_arch + 1                          # colorbars below the spacer row
cax0 = fig.add_subplot(gs[cbrow, 2])
cb0 = fig.colorbar(ScalarMappable(Normalize(a_lo, a_hi), 'cividis'), cax=cax0, orientation='horizontal')
cb0.ax.tick_params(labelsize=6); cb0.set_label('ATAC norm log2-CPM', fontsize=6)
cax1 = fig.add_subplot(gs[cbrow, 2 + len(panel_cols) - 1])
cb1 = fig.colorbar(ScalarMappable(Normalize(-ZLIM, ZLIM), 'RdBu_r'), cax=cax1, orientation='horizontal')
cb1.ax.tick_params(labelsize=6); cb1.set_label('histone column z-score', fontsize=6)
fig.suptitle('DA cCRE archetypes — histone sub-clusters (EVoC), per-cluster tiles', fontsize=11, y=0.995)

fig.savefig(FIG_DIR / 'histsub_tiled.png', dpi=200, bbox_inches='tight')
fig.savefig(FIG_DIR / 'histsub_tiled.pdf')

## Figure 2 — sub-cluster dendrograms (per archetype)

In [ ]:
# precompute linkages so all panels can share a common Ward-distance y-scale
linkages = {}
for cl in range(1, n_arch + 1):
    m = arch == cl
    subs = sorted(s for s in np.unique(sub[m]) if s != 99)
    if len(subs) < 2:
        continue
    cents, labels = [], []
    for s in subs:
        sel = m & (sub == s)
        c = histz[sel].mean(0); cents.append(c)
        blk = c.reshape(len(HIST), len(STAGES)).mean(1)
        dom = HIST[int(np.argmax(np.abs(blk)))]
        labels.append(f"s{s} (n={int(sel.sum())}, {dom}{'+' if blk[np.argmax(np.abs(blk))]>0 else '-'})")
    linkages[cl] = (linkage(np.array(cents), method='ward'), labels, len(subs))
ymax = 1.05 * max(Z[:, 2].max() for Z, _, _ in linkages.values())

fig, axes = plt.subplots(2, 4, figsize=(18, 9), sharey=True)
for cl, ax in zip(range(1, n_arch + 1), axes.flat):
    if cl not in linkages:
        ax.set_title(f'archetype {cl}: <2 sub-states'); ax.axis('off'); continue
    Z, labels, nsub = linkages[cl]
    dendrogram(Z, labels=labels, ax=ax, leaf_rotation=90, leaf_font_size=7,
               link_color_func=lambda _: '0.4')
    for lbl in ax.get_xticklabels():
        sid = int(lbl.get_text().split('(')[0].strip().lstrip('s'))
        lbl.set_color(TAB10[sid % 10])
    ax.set_ylim(0, ymax)                 # common y-scale across archetypes
    ax.set_title(f'archetype {cl}  ({nsub} histone sub-states)', fontsize=10)
    ax.set_ylabel('Ward distance', fontsize=8)
fig.suptitle('Histone sub-state dendrograms within each ATAC archetype '
             '(Ward linkage on 4-histone column-z centroids; common y-scale)', fontsize=12)
fig.tight_layout()

fig.savefig(FIG_DIR / 'histsub_dendro.png', dpi=140, bbox_inches='tight')
fig.savefig(FIG_DIR / 'histsub_dendro.pdf', bbox_inches='tight')

## Figure 3 — fuzzy c-means profile plots (datashader)

In [ ]:
memb = df['membership'].to_numpy()
n_cl = int(arch.max())
ylo, yhi = np.percentile(atac, [0.5, 99.5])
span = (float(memb.min()), float(memb.max()))
mnorm = Normalize(*span)
xs = np.arange(len(STAGES), dtype='float64')      # np array -> shared-x line glyph
CIVIDIS = [mcolors.rgb2hex(plt.cm.cividis(i / 255)) for i in range(256)]
PW, PH = 900, 600

fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True, sharey=True)
for c, ax in zip(range(1, n_cl + 1), axes.flat):
    idx = np.where(arch == c)[0]
    T = atac[idx]; U = memb[idx]
    cen = (U[:, None] * T).sum(0) / U.sum()
    peak = STAGES[int(cen.argmax())]
    pdf = pd.DataFrame(T, columns=STAGES); pdf['membership'] = U
    cvs = ds.Canvas(plot_width=PW, plot_height=PH,
                    x_range=(0, len(STAGES) - 1), y_range=(ylo, yhi))
    agg = cvs.line(pdf, x=xs, y=STAGES, agg=ds.mean('membership'), axis=1)
    img = tf.shade(agg, cmap=CIVIDIS, how='linear', span=span).to_pil()
    ax.imshow(np.asarray(img), extent=[0, len(STAGES) - 1, ylo, yhi],
              origin='upper', aspect='auto', interpolation='nearest')
    ax.plot(xs, cen, color='white', lw=4, zorder=4)
    ax.plot(xs, cen, color='black', lw=2.5, zorder=5)
    ax.set_xlim(0, len(STAGES) - 1); ax.set_ylim(ylo, yhi)
    ax.set_xticks(xs); ax.set_xticklabels(STAGES, fontsize=8)
    ax.set_title(f'cl{c}  (n={len(idx):,}, peak@{peak})', fontsize=10)
    ax.axhline(0, color='0.8', lw=0.5, zorder=0)
for ax in axes[:, 0]:
    ax.set_ylabel('ATAC norm log2-CPM', fontsize=9)
cb = fig.colorbar(ScalarMappable(mnorm, 'cividis'), ax=axes, fraction=0.02, pad=0.01)
cb.set_label('fuzzy membership (mean per pixel)', fontsize=9)
fig.suptitle('ATAC archetypes — fuzzy c-means profiles (datashader; all members; '
             'black = weighted centroid)', fontsize=12, y=0.98)

fig.savefig(FIG_DIR / 'histsub_profiles.png', dpi=150, bbox_inches='tight')
fig.savefig(FIG_DIR / 'histsub_profiles.pdf', bbox_inches='tight')